[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/entropy_production/entropy_production.ipynb)

# Entropy Production in Heat Exchange

This example computes how two insulated bodies at different temperatures exchange heat until their temperatures equalize. Heat flows from the hotter body to the colder one while keeping their total energy constant. Because each unit of energy leaves at a higher temperature than the temperature at which it arrives, total entropy rises until the pair reaches the state of maximum entropy for that total energy.

The algebra `a0 b0` gives each body's internal energy its own degenerate quantity direction. A state is modelled as an antivector `energy_a * a.dual() + energy_b * b.dual()`, and a thermodynamic potential is a vector that pairs with changes of state. The entropic potential `a / temperature_a + b / temperature_b` pairs with a change of state to give the entropy produced, while `a + b` reads the change in total energy. The mobility maps potentials to rates of state change through a positive dyad `transfer * (Potential & transfer)` along `transfer = b.dual() - a.dual()`, the step that moves a unit of energy from body `a` to body `b`. Because `a + b` pairs to zero with `transfer`, the flow driven by the entropic potential conserves total energy and produces nonnegative entropy.

In the notation of ordinary differential equations, with a dot for the time derivative, the temperatures follow Newton's law of cooling, $C_a \dot T_a = -k(T_a - T_b) = -C_b \dot T_b$, and entropy is produced at the rate $\dot S = k(T_a - T_b)^2 / (T_a T_b) \ge 0$. In the matrix notation of the GENERIC formulation, this dissipative flow reads $\dot x = M\,\nabla S$, where the symmetric positive semidefinite mobility matrix $M$ satisfies $M\,\nabla E = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.thermodynamics.entropy_production import render
from examples.util.animation import show_states

ga = Algebra("a0b0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                               # Scalar+
State = ga.gatype.antivector()                            # Antivector-
Potential = ga.gatype.vector()                            # Vector+
Mobility = ga.gatype((State, Potential))                  # State- <- Potential+

energy_a = mv.a & State                                   # [] Scalar+ <- State-
energy_b = mv.b & State                                   # [] Scalar+ <- State-
# Total-energy potential and the energy-conserving unit transfer from body a to body b, which pair to zero.
energy_potential = mv.a + mv.b                            # [] Potential+
transfer = mv.b.dual() - mv.a.dual()                      # [] State-
# Unit mobility dyad sending any potential along transfer in proportion to how that potential reads transfer.
unit_mobility = transfer * (Potential & transfer)         # [] State- <- Potential+

## Entropic potential and mobility

Each body has constant heat capacity, so its temperature is its internal energy divided by its heat capacity and its entropy grows with the logarithm of its temperature, measured against a unit reference temperature. Scaling `unit_mobility` by the bar's conductance and both temperatures gives a heat flow proportional to the temperature difference, matching Fourier conduction across the bar while keeping the mobility positive at every positive temperature.

In [ ]:
capacity_a, capacity_b, conductance = 1.0, 2.0, 0.6
capacities = np.array([capacity_a, capacity_b])

temperature_a = energy_a / capacity_a                                                       # [] Scalar+ <- State-
temperature_b = energy_b / capacity_b                                                       # [] Scalar+ <- State-
temperatures = stack((temperature_a, temperature_b))                                        # [2] Scalar+ <- State-


def entropy(state: State) -> Scalar:
    """Total entropy of the two bodies relative to a unit reference temperature."""
    return capacity_a * temperature_a(state).log() + capacity_b * temperature_b(state).log()


def entropy_potential(state: State) -> Potential:
    """Thermodynamic potential that pairs with a change of state to give the entropy produced."""
    return mv.a / temperature_a(state) + mv.b / temperature_b(state)


def mobility(state: State, bar_conductance: float) -> Mobility:
    """Positive dyad mapping potentials to energy-conserving rates of state change."""
    return unit_mobility * (bar_conductance * temperature_a(state) * temperature_b(state))


initial_temperature_a, initial_temperature_b = 2.8, 0.6
initial_state = (
    mv.a.dual() * (capacity_a * initial_temperature_a)
    + mv.b.dual() * (capacity_b * initial_temperature_b)
)                                                                                           # [] State-
initial_flow = mobility(initial_state, conductance)(entropy_potential(initial_state))       # [] State-
render.draw(entropy, entropy_potential, temperatures, transfer, capacities, initial_state);

In [ ]:
np.set_printoptions(precision=4, suppress=True)
print("initial energy rates:", (mv.a & initial_flow).to_array(), (mv.b & initial_flow).to_array())
print("initial entropy production rate:", (entropy_potential(initial_state) & initial_flow).to_array())

## Heat exchange over time

Because the heat flow is proportional to the temperature difference, that difference decays exponentially with relaxation time `1 / (conductance * (1 / capacity_a + 1 / capacity_b))`. Integrating the decay over a time step `dt` gives one exact step of `mobility` evaluated with an effective integrated conductance.

In [ ]:
inverse_capacity = 1 / capacity_a + 1 / capacity_b
relaxation_time = 1 / (conductance * inverse_capacity)
n_frames, n_relaxation_times, frame_duration_ms = 135, 5, 40
dt = n_relaxation_times * relaxation_time / (n_frames - 1)


def advance(state: State, step_dt: float) -> State:
    """State after integrating the heat exchange analytically over `step_dt`."""
    integrated_conductance = -np.expm1(-conductance * inverse_capacity * step_dt) / inverse_capacity
    return state + mobility(state, integrated_conductance)(entropy_potential(state))


def exchange() -> Iterator[tuple[State, float]]:
    """Yield the state and elapsed time at each frame."""
    state = initial_state
    for frame in range(n_frames):
        yield state, frame * dt
        state = advance(state, dt)


frames = show_states(
    exchange(), render.exchange, entropy, entropy_potential, temperatures, transfer, capacities,
    name="entropy_production", duration_ms=frame_duration_ms, n_frames=n_frames,
)

Each block is scaled by its heat capacity and coloured by its temperature gap from equilibrium, blending the gap with its logarithm so the colours keep changing as the temperatures converge. In the first diagram, teal curves mark contours of constant entropy on the plane of internal energies, and the state moves from hot body `a` toward cold body `b` along the black line of constant total energy until it meets the black tangent entropy contour at the green equilibrium dot. The middle chart plots entropy along that constant-energy line against the energy of body `a`, with the orange tangent showing the entropic slope that drives the state to the peak. The right chart tracks both temperatures over time as they relax toward their equilibrium temperature of `4 / 3`, the total energy divided by the total heat capacity.

In [ ]:
states = stack([state for state, _ in frames])                                              # [n_frames] State-
final_temperatures = temperatures(states[-1])                                               # [2] Scalar+
print("final temperatures:", final_temperatures.to_array())
print("relaxation time:", round(relaxation_time, 4))

In [ ]:
# checks
entropies = entropy(states)                                                                 # [n_frames] Scalar+
np.testing.assert_allclose(((energy_potential & states) - (energy_potential & initial_state)).kernel, 0, atol=1e-6)
assert np.all(entropies[1:] - entropies[:-1] >= -1e-6)
np.testing.assert_allclose(mobility(initial_state, conductance)(energy_potential).kernel, 0, atol=1e-6)
np.testing.assert_allclose((final_temperatures - 4 / 3).kernel, 0, atol=1e-2)